In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.models.struggle_predictor import StrugglePredictor
from src.models.failure_detector import detect_one, FAILURE_TYPES
from src.simulator.student_simulator import StudentSimulator, REPAIR_ACTIONS

print("Loading TSRP and simulator...")
tsrp = StrugglePredictor()
sim = StudentSimulator(rng=np.random.default_rng(123))
print("OK.")

Loading TSRP and simulator...
OK.


In [2]:
obs = sim.reset()
print("Initial observation:")
for k, v in obs.items():
    print(f"  {k}: {v}")

Initial observation:
  user_id: 70752
  skill_id: 280
  skill_correct_rate: 1.0
  help_dependency: 0.0
  recent_correct_rate: 1.0
  prev_correct: 1
  prev_was_repair_event: 1
  repair_need_score: 0.11462306283491713
  effort_cost_score: 1.3080812256376093
  disengagement_risk: 0.17706867804612025
  failure_type: repair_needed_failure
  true_correct: 0
  true_hint: 2
  true_attempts: 6


In [3]:
# The simulator already attached these (precomputed). Let's verify
# we can also recompute the struggle vector live from raw features.
# For the live recomputation we need feature columns. Pull from the
# underlying parquet for this user's current row.

row = sim.current_user_rows.iloc[sim.current_step]
feature_dict = {c: row[c] for c in tsrp.feature_cols_full}
live_pred = tsrp.predict(feature_dict)

print("Stored struggle vector (precomputed):")
print(f"  repair_need={obs['repair_need_score']:.3f}, "
      f"effort={obs['effort_cost_score']:.3f}, "
      f"diseng={obs['disengagement_risk']:.3f}")
print("Live TSRP prediction (recomputed):")
print(f"  repair_need={live_pred['repair_need_score']:.3f}, "
      f"effort={live_pred['effort_cost_score']:.3f}, "
      f"diseng={live_pred['disengagement_risk']:.3f}")
print(f"\nFailure type: {obs['failure_type']}")

Stored struggle vector (precomputed):
  repair_need=0.115, effort=1.308, diseng=0.177
Live TSRP prediction (recomputed):
  repair_need=0.115, effort=1.308, diseng=0.177

Failure type: repair_needed_failure


In [4]:
log = []
obs = sim.reset()
for t in range(10):
    action = REPAIR_ACTIONS[np.random.default_rng(t).integers(len(REPAIR_ACTIONS))]
    next_obs, info = sim.step(action)
    log.append({
        "turn": t,
        "action": action,
        "failure_before": info["failure_type_before"],
        "sim_correct": info["sim_correct"],
        "sim_hint": info["sim_hint"],
        "shifted_p_correct": round(info["shifted_correct_p"], 3),
    })
    if info["done"]:
        break

pd.DataFrame(log)

,turn,action,failure_before,sim_correct,sim_hint,shifted_p_correct
0,0,simpler_explanation,repair_needed_failure,0,0,0.04


In [5]:
from collections import Counter

cnt = Counter()
n_episodes = 30
n_skipped = 0
sim2 = StudentSimulator(rng=np.random.default_rng(0))

for ep in range(n_episodes):
    obs = sim2.reset_with_min_failures(min_failures=3)
    if obs is None:
        n_skipped += 1
        continue
    # Count the starting failure type
    cnt[obs["failure_type"]] += 1
    for t in range(20):
        a = REPAIR_ACTIONS[np.random.default_rng(ep * 100 + t).integers(len(REPAIR_ACTIONS))]
        next_obs, info = sim2.step(a)
        if info.get("skipped"):
            continue
        if info["done"]:
            break
        # info["failure_type_before"] is the row the action acted on
        cnt[info["failure_type_before"]] += 1

print(f"Episodes run: {n_episodes - n_skipped}, skipped: {n_skipped}")
print("\nFailure type encounters (only failure rows, no leakage):")
for k, v in cnt.most_common():
    print(f"  {k}: {v}")

Episodes run: 30, skipped: 0

Failure type encounters (only failure rows, no leakage):
  repair_needed_failure: 327
  low_mastery_failure: 203
